# Unitary means reversible
**Quantum Computing · Week 3 (The maths of qubits, and interference) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qc/w3d2.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

A gate must keep the total probability equal to 1 for every input. That single rule, **unitarity**,
makes every gate reversible.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

def dagger(U):
    """The conjugate transpose U†."""
    return U.conj().T

s = 1 / np.sqrt(2)
gates = {
    "H": np.array([[s, s], [s, -s]]),
    "S": np.array([[1, 0], [0, 1j]]),
    "T": np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]]),
    "RY(40°)": np.array([[np.cos(np.radians(20)), -np.sin(np.radians(20))],     # RY(θ) uses θ/2 = 20°
                         [np.sin(np.radians(20)), np.cos(np.radians(20))]]),
}
for name, U in gates.items():
    print(f"{name:>8}: U†U = I ? {np.allclose(dagger(U) @ U, np.eye(2))}")

## What unitarity means: lengths and right angles
The columns of a unitary have length 1 and are perpendicular (their inner product is 0).
So the gate maps the "perpendicular" states $|0\rangle, |1\rangle$ to two perpendicular states.

In [ ]:
for name, U in gates.items():
    c0, c1 = U[:, 0], U[:, 1]
    print(f"{name:>8}: |col0| = {np.linalg.norm(c0):.3f}, |col1| = {np.linalg.norm(c1):.3f}, <col0|col1> = {abs(np.vdot(c0, c1)):.3f}")

## Not every matrix is allowed
This matrix sends both $|0\rangle$ and $|1\rangle$ to $|0\rangle$: information is lost, and the total probability breaks.

In [ ]:
M = np.array([[1, 1], [0, 0]])
plus = np.array([s, s])
out = M @ plus
print("M† M =\n", dagger(M) @ M)
print("M |+> =", np.round(out, 3), "  total probability =", round(np.sum(np.abs(out) ** 2), 3), "(should be 1)")

## Undoing a circuit
`qc.inverse()` reverses the gates and replaces each one by its inverse. A circuit followed by its inverse does nothing.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.t(0)
qc.ry(0.7, 0)
undo = qc.inverse()
print("circuit:\n", qc.draw("text"))
print("its inverse:\n", undo.draw("text"))
both = qc.compose(undo)
print("circuit + inverse = identity?", Operator(both).equiv(Operator(np.eye(2))))
print("state at the end:", np.round(Statevector(both).data, 3) + 0)

## Your turn: undo by hand
Build the inverse of `steps` yourself: go through the steps **backwards** and invert each one
(`h` stays `h`, `s` becomes `sdg`, `t` becomes `tdg`, a rotation by θ becomes a rotation by -θ).

In [ ]:
steps = [("rx", 0.5), ("h", None), ("s", None), ("rz", 1.1), ("t", None)]
qc = QuantumCircuit(1)
for name, angle in steps:
    getattr(qc, name)(*([angle] if angle is not None else []), 0)

mine = QuantumCircuit(1)
# TODO: loop over reversed(steps) and add the inverse of each step to `mine`

print("mine undoes it?", Operator(qc.compose(mine)).equiv(Operator(np.eye(2))))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
inverse_name = {"h": "h", "s": "sdg", "t": "tdg", "x": "x"}
mine = QuantumCircuit(1)
for name, angle in reversed(steps):
    if angle is None:
        getattr(mine, inverse_name[name])(0)
    else:
        getattr(mine, name)(-angle, 0)          # rx(θ) is undone by rx(-θ)
print(mine.draw("text"))
print("mine undoes it?", Operator(qc.compose(mine)).equiv(Operator(np.eye(2))))
print("same as qc.inverse()?", Operator(mine).equiv(Operator(qc.inverse())))